# <center> Funcomics from DEA

## <center> Description

- env: squidpy_v09

## <center> Setup

#### **Python**

In [1]:
# Load libraries
import numpy as np
import pandas as pd
import decoupler as dc
import scanpy as sc
from scipy import sparse

## plots
from math import ceil
import matplotlib.pyplot as plt
import plotnine as p9
import seaborn as sns

# Add the project root to sys.path so local workflow.* imports work both in
# notebooks (where __file__ is undefined) and in exported Python scripts.
import sys
from pathlib import Path
location = globals().get("__file__") or globals().get("__vsc_ipynb_file__") or Path.cwd()
location = Path(location).resolve()
start_dir = location if location.is_dir() else location.parent
PROJECT_ROOT = next(
    (path for path in [start_dir, *start_dir.parents] if (path / "workflow/Snakefile").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(f"Could not find the project root above {start_dir}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## helper functions
from workflow.scripts.funs.helper_funs import get_project_root

#### **Paths**

In [2]:
if 'snakemake' in locals():
    print('Hello World')
else:
    ## get paths
    ROOT = get_project_root(globals())
    
    # input
    ## DEA results (above)
    in_deg = ROOT / "output/DEA/objects/calibrate_pvalues_interaction/deg_calibrated.tsv"
    in_adata = ROOT / "output/preprocessing/objects/QC/adata.h5ad"
    ## funcomics
    path_input_objects = ROOT / "output/preprocessing/objects/getGenesets"
    in_collectri=path_input_objects / 'collectri.tsv'
    in_progeny=path_input_objects / 'progeny.tsv'
    in_go_cc=path_input_objects / 'go_cc.tsv'
    in_go_mf=path_input_objects / 'go_mf.tsv'
    in_go_bp=path_input_objects / 'go_bp.tsv'
    in_hallmark=path_input_objects / 'hallmark.tsv'
    in_reactome=path_input_objects / 'reactome.tsv'
    in_KEGG=path_input_objects / 'kegg.tsv'

    # output
    ## plots
    out_plots = ROOT / "output/DEA/plots/funcomics_ulm"
    out_plots.mkdir(parents=True, exist_ok=True)
    ## objects
    out_dea_enrich=ROOT / "output/DEA/objects/funcomics_ulm/funcomics.tsv"
    out_dea_enrich.parent.mkdir(parents=True, exist_ok=True) # make sure the output folder exists
    ### tables with data to reproduce each figure
    out_tab_progeny = ROOT / "output/DEA/objects/funcomics_ulm/Fig5F_progeny_overview.tsv"
    out_tab_collectri = ROOT / "output/DEA/objects/funcomics_ulm/Fig5E_collectri_overview.tsv"
    out_tab_collectri_IRF1 = ROOT / "output/DEA/objects/funcomics_ulm/Fig5G_IRF1.tsv"
    out_tab_collectri_NFKB1 = ROOT / "output/DEA/objects/funcomics_ulm/Fig5G_NFKB1.tsv"
    out_tab_progeny_hypoxia = ROOT / "output/DEA/objects/funcomics_ulm/Fig5G_hypoxia.tsv"
    out_tab_progeny_NFKB = ROOT / "output/DEA/objects/funcomics_ulm/Fig5G_NFKB.tsv"
    out_tab_progeny_TNFa = ROOT / "output/DEA/objects/funcomics_ulm/Fig5G_TNFa.tsv"
    out_tab_progeny_JAKSTAT = ROOT / "output/DEA/objects/funcomics_ulm/Fig5G_JAKSTAT.tsv"
    out_tab_customGenes = ROOT / "output/DEA/objects/funcomics_ulm/Fig5G_customGenes.tsv"

In [3]:
colname_comparison='contrast'
colname_features='genesymbol'
## stat will be used instead of calibrated padj because almost all stat vals are unique
## to the contrary, there are only 216 / 19k unique calibrated padj values! 
## GSEA relies on ranking.
colname_stat='stat' 

## <center> Load data

#### **adata**

In [4]:
adata = sc.read_h5ad(in_adata)
## remove B samples
adata=adata[adata.obs['QC_group']=='A'].copy()
## remove genes with missing genesymbol ID
adata=adata[:,~adata.var['genesymbol'].isna()].copy()
## remove genes with duplicated genesymbol ID (11 genes)
duplicated_genes=adata.var['genesymbol'][adata.var['genesymbol'].duplicated()].unique().tolist()
adata=adata[:,~adata.var['genesymbol'].isin(duplicated_genes)].copy()
## genesymbol as new index
adata.var.index=adata.var['genesymbol'].copy()
## use deseq2 normalized counts
adata.X = adata.layers['cts.deseq2_norm'].copy()

adata

AnnData object with n_obs × n_vars = 50 × 17846
    obs: 'disease_group', 'QC_group', 'condition_group', 'grp', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.2'
    var: 'genesymbol', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'QC_group_colors', 'condition_group_colors', 'disease_group_colors', 'grp_colors', 'hvg', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.15_colors', 'leiden_0.2', 'log1p', 'neighbors', 'pca', 'rank_obsm', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'cts.deseq2_norm', 'cts.deseq2_norm_log', 'cts.raw', 'cts.vst'
    obsp: 'connectivities', 'distances'

In [5]:
# add individualID

col_dict = {
    'SD1': 'individual_01','SD2': 'individual_01','SD3': 'individual_02','SD4': 'individual_02',
    'SD5': 'individual_03','SD6': 'individual_03','SD7': 'individual_04','SD8': 'individual_04',
    'SD9': 'individual_05','SD10': 'individual_05','SD11': 'individual_06','SD12': 'individual_06',
    'SD13': 'individual_07','SD14': 'individual_07','SD15': 'individual_08','SD16': 'individual_08',
    'SD17': 'individual_09','SD18': 'individual_09','SD19': 'individual_10','SD20': 'individual_10',
    'SD21': 'individual_11','SD22': 'individual_11','SD23': 'individual_12','SD24': 'individual_12',
    'SD25': 'individual_13','SD26': 'individual_13','SD27': 'individual_14','SD28': 'individual_14',
    'SD29': 'individual_15','SD30': 'individual_15','SD31': 'individual_16','SD32': 'individual_16',
    'SD33': 'individual_17','SD34': 'individual_17','SD35': 'individual_18','SD36': 'individual_18',
    'SD37': 'individual_19','SD38': 'individual_19','SD39': 'individual_20','SD40': 'individual_20',
    'SD41': 'individual_21','SD42': 'individual_21','SD43': 'individual_22','SD44': 'individual_22',
    'SD45': 'individual_23','SD46': 'individual_23','SD47': 'individual_24','SD48': 'individual_24',
    'SD49': 'individual_25','SD50': 'individual_25','SD51': 'individual_26','SD52': 'individual_26',
    'SD53': 'individual_27','SD54': 'individual_27','SD55': 'individual_28','SD56': 'individual_28',
    'SD57': 'individual_29','SD58': 'individual_29','SD59': 'individual_30','SD60': 'individual_30',
    'SD61': 'individual_31','SD62': 'individual_31','SD63': 'individual_32','SD64': 'individual_32',
    'SD65': 'individual_33','SD66': 'individual_33',
}
adata.obs['individualID']=adata.obs.index.map(col_dict)
adata.obs['individualID']=adata.obs['individualID'].astype('category')

In [6]:
# get fold change
## by individualID, IFNG+ / IFNG-


## get expression
df_log2fc = pd.DataFrame(adata.layers['cts.deseq2_norm'], index=adata.obs.index,columns=adata.var.index)
df_log2fc = (
    df_log2fc.reset_index().
    melt(id_vars=['sample_id'],var_name='genesymbol',value_name='expr'))
## add individualID
col_dict=adata.obs['individualID'].to_dict()
df_log2fc['individualID']=df_log2fc['sample_id'].map(col_dict)
## add other columns
cols=['condition_group']
for col in cols:
    col_dict=adata.obs[col].to_dict()
    df_log2fc[col]=df_log2fc['sample_id'].map(col_dict)
## get fold change
df_log2fc = (
    df_log2fc
    .pivot(index=["individualID", "genesymbol"], columns="condition_group", values="expr")
    .reset_index()
)
eps = 1e-6
df_log2fc["fold_change"] = (df_log2fc["IFNG_pos"] + eps) / (df_log2fc["IFNG_neg"] + eps)
df_log2fc["log2_fold_change"] = np.log2(df_log2fc["fold_change"])

df_log2fc.head()

condition_group,individualID,genesymbol,IFNG_neg,IFNG_pos,fold_change,log2_fold_change
0,individual_01,5_8S_rRNA,119.798671,106.803051,0.891521,-0.165659
1,individual_01,7SK,27.645847,25.075499,0.907026,-0.140784
2,individual_01,A1BG,142.324916,123.520051,0.867874,-0.204443
3,individual_01,A1BG-AS1,104.439867,121.662606,1.164906,0.220213
4,individual_01,A1CF,0.000000,0.928722,928723.185942,19.824889


In [7]:
# # get a table with log2FC values for all genes

# ## pseudocount prevents division by zero
# pseudocount = 1.0

# log2fc_rows = {}
# for individual_id, obs in adata.obs.groupby("individualID", observed=True):
#     pos_idx = obs.index[obs["condition_group"] == "IFNG_pos"]
#     neg_idx = obs.index[obs["condition_group"] == "IFNG_neg"]

#     # Skip individuals missing either condition.
#     if len(pos_idx) == 0 or len(neg_idx) == 0:
#         continue

#     pos_mean = adata[pos_idx].X.mean(axis=0)
#     neg_mean = adata[neg_idx].X.mean(axis=0)

#     if sparse.issparse(pos_mean):
#         pos_mean = pos_mean.A1
#         neg_mean = neg_mean.A1
#     else:
#         pos_mean = np.asarray(pos_mean).ravel()
#         neg_mean = np.asarray(neg_mean).ravel()

#     log2fc_rows[individual_id] = np.log2(
#         (pos_mean + pseudocount) / (neg_mean + pseudocount)
#     )

# # Rows: individuals; columns: genes
# df_log2fc = pd.DataFrame.from_dict(
#     log2fc_rows,
#     orient="index",
#     columns=adata.var_names,
# )
# df_log2fc.index.name = "individualID"

# df_log2fc.head()

#### **degs**

In [8]:
deg=pd.read_csv(in_deg,sep='\t')
## filter to genes in adata
deg=deg[deg['genesymbol'].isin(adata.var.index)].copy()
# ## add a new col sig
# deg['sig']=-np.log10(deg['padj'])
# deg['sig']=[sig*1 if FC>1 else sig*-1 for sig,FC in zip(deg['sig'],deg['log2FoldChange'])]
## simplify 'contrast' col
deg['contrast']='interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos'

deg.head()

,gene_id,genesymbol,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,contrast,design_formula,coefficient_tested,significant_pvalue_0.05,significant_padj_0.05,direction,pvalue_raw,padj_raw,null_sd
0,ENSG00000183486,MX2,45.033753,1.798912,0.466584,3.855496,0.000014,0.222899,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,~ individualID + ifng_pos + case_ifng_pos,case_ifng_pos,True,False,stronger_IFNG_response_in_HR_case,0.000115,0.99962,0.886861
1,ENSG00000137752,CASP1,84.326608,3.247200,0.865542,3.751640,0.000023,0.222899,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,~ individualID + ifng_pos + case_ifng_pos,case_ifng_pos,True,False,stronger_IFNG_response_in_HR_case,0.000176,0.99962,0.886861
2,ENSG00000158258,CLSTN2,5745.902787,-0.357196,0.098084,-3.641733,0.000040,0.222899,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,~ individualID + ifng_pos + case_ifng_pos,case_ifng_pos,True,False,stronger_IFNG_response_in_HR_control,0.000271,0.99962,0.886861
3,ENSG00000140807,NKD1,1714.670842,-0.348793,0.097321,-3.583956,0.000053,0.222899,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,~ individualID + ifng_pos + case_ifng_pos,case_ifng_pos,True,False,stronger_IFNG_response_in_HR_control,0.000338,0.99962,0.886861
4,ENSG00000182568,SATB1,215.111114,-0.423812,0.119857,-3.535986,0.000067,0.222899,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,~ individualID + ifng_pos + case_ifng_pos,case_ifng_pos,True,False,stronger_IFNG_response_in_HR_control,0.000406,0.99962,0.886861


## <center> Funcomics

#### **Load gene sets**

In [9]:
# Load gene sets

# collectri TFs
df_collectri=pd.read_csv(in_collectri, sep='\t')
# progeny
df_progeny=pd.read_csv(in_progeny, sep='\t')
# go_bp
df_go_bp=pd.read_csv(in_go_bp, sep='\t').rename(columns={'geneset':'source', 'genesymbol':'target'})
# go_mf
df_go_mf=pd.read_csv(in_go_mf, sep='\t').rename(columns={'geneset':'source', 'genesymbol':'target'})
# go_cc
df_go_cc=pd.read_csv(in_go_cc, sep='\t').rename(columns={'geneset':'source', 'genesymbol':'target'})
# hallmark
df_hallmark=(
    pd.read_csv(in_hallmark, sep='\t').
    rename(columns={'geneset':'source', 'genesymbol':'target'}).
    assign(source=lambda x:x['source'].str.replace('HALLMARK','HK'))
)
# KEGG
df_kegg=(
    pd.read_csv(in_KEGG, sep='\t').
    rename(columns={'geneset':'source', 'genesymbol':'target'}).
    assign(source=lambda x:x['source'].str.replace('KEGG','KG'))
)
# reactome
df_reactome=(
    pd.read_csv(in_reactome, sep='\t').
    rename(columns={'geneset':'source', 'genesymbol':'target'}).
    assign(source=lambda x:x['source'].str.replace('REACTOME','RE'))
)

## From DEG results

#### **Prepare data**

In [10]:
# prepare input data

mat=deg.pivot_table(index=colname_comparison, columns=colname_features, values=colname_stat)

mat.head()

genesymbol,5_8S_rRNA,7SK,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A4GALT,AAAS,AACS,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
contrast,,,,,,,,,,,,,,,,,,,,,
interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,-0.013059,-0.964802,-0.223895,-0.472813,0.219543,1.336566,0.066378,-0.050272,0.889471,-0.284105,...,0.942663,-0.464768,1.29885,-0.126885,0.353752,-0.767206,-0.353744,0.465053,-2.043893,0.848965


#### **Perform enrichment**

In [11]:
# res_lst=[]
# input_lst=[
#     ('collectri',df_collectri),('progeny',df_progeny),
#     ('go_bp',df_go_bp),('go_mf',df_go_mf),('go_cc',df_go_cc),
#     ('hallmark',df_hallmark),('kegg',df_kegg),('reactome',df_reactome)]
# for pk,df_input in input_lst:
#     print(pk)
#     ## get results
#     res_acts, res_p = dc.mt.gsea(data=mat,net=df_input,verbose=False,times=500,seed=0)
#     ## merge act/pvalue results
#     res=(
#         res_acts.reset_index(names='comparison').
#         melt(id_vars='comparison', var_name='source', value_name='act').
#         merge(
#             res_p.reset_index(names='comparison').melt(id_vars='comparison', var_name='source', value_name='padj')).
#         assign(gene_set=pk).
#         sort_values(by='padj', ascending=True))
#     ## save results
#     res_lst.append(res)
# # concat
# df_enrich=pd.concat(res_lst)
# # save
# df_enrich.to_csv(out_dea_enrich, sep='\t', index=False)

# df_enrich.head()

In [12]:
# df_enrich[df_enrich['padj']<0.05].groupby('gene_set').size()

In [13]:
res_lst=[]
input_lst=[
    ('collectri',df_collectri),('progeny',df_progeny),
    ('go_bp',df_go_bp),('go_mf',df_go_mf),('go_cc',df_go_cc),
    ('hallmark',df_hallmark),('kegg',df_kegg),('reactome',df_reactome)]
for pk,df_input in input_lst:
    ## get results
    res_acts, res_p = dc.mt.ulm(data=mat,net=df_input,verbose=False)
    ## merge act/pvalue results
    res=(
        res_acts.reset_index(names='comparison').
        melt(id_vars='comparison', var_name='source', value_name='act').
        merge(
            res_p.reset_index(names='comparison').melt(id_vars='comparison', var_name='source', value_name='padj')).
        assign(gene_set=pk).
        sort_values(by='padj', ascending=True))
    ## save results
    res_lst.append(res)
# concat
df_enrich=pd.concat(res_lst)
# save
df_enrich.to_csv(out_dea_enrich, sep='\t', index=False)

df_enrich.head()

,comparison,source,act,padj,gene_set
258,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,IRF1,5.306866,0.000080,collectri
392,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,NFKB1,4.577813,0.001686,collectri
507,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,RELA,3.618368,0.063293,collectri
124,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,ETS2,3.572089,0.063293,collectri
61,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,CREBZF,-3.497256,0.067195,collectri


In [14]:
# how many significantly enriched by gene-set?
df_enrich[df_enrich['padj']<0.05].groupby('gene_set').size()

gene_set
collectri      2
go_bp        227
go_cc         88
go_mf         96
hallmark      16
kegg          16
progeny        5
reactome      57
dtype: int64

## <center> Plots

## Overview of enrichment

#### **CollecTRI**

In [15]:
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[(plotTab['gene_set']=='collectri')]
## select top TFs
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']=[-np.log10(p) * 1 if act>1 else -np.log10(p) * -1 for act,p in zip(plotTab['act'],plotTab['padj'])]
## order by activity
plotTab=plotTab.sort_values(by='act',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

plotTab.head()

,comparison,source,act,padj,gene_set,sig
61,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,CREBZF,-3.497256,0.067195,collectri,-1.172662
563,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,SOX4,-3.382690,0.085481,collectri,-1.068129
364,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,NAB2,-3.023825,0.198020,collectri,-0.703290
273,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,JUND,2.957578,0.221367,collectri,0.654887
574,interaction_HRcaseIFNgpos_vs_HRctrlIFNgpos,SPI1,3.077740,0.186176,collectri,0.730076


In [16]:
# barplot


h,w=3,6

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='act'),color='black', stat='identity', show_legend=True,alpha=0.7) +
    p9.scale_fill_gradient2(low="#2166AC",mid="white",high="#B2182B",midpoint=0) +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.geom_hline(yintercept=np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_classic() +
    p9.labs(
        title="Top collecTRI TFs (ulm)",
        subtitle='contrast: differential IFNG response (+,-) across disease groups (HR cases,HR controls) (interaction)\ndashed lines: padj 0.05',
        y='-log10(padj) * direction',
        fill='TF\nactivity'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        axis_line_y=p9.element_blank(), axis_ticks_major_y=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / "collectri_overview.png", width=w, height=h, limitsize=False)
p.save(out_plots / "collectri_overview.pdf", width=w, height=h, limitsize=False)
plt.close()

/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 3 in image.
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /Users/miguel/Documents/Saezlab/project/olabisi_repo/olabisi/output/DEA/plots/funcomics_ulm/collectri_overview.png
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 3 in image.
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /Users/miguel/Documents/Saezlab/project/olabisi_repo/olabisi/output/DEA/plots/funcomics_ulm/collectri_overview.pdf


In [17]:
# save table with data to recreate plot

plotTab.drop(columns=['comparison']).rename(columns={'sig':'neglog10_padj_x_direction'}).to_csv(out_tab_collectri,sep='\t')

#### **Progeny**

In [18]:
# barplot

pk='progeny'
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[plotTab['gene_set']==pk]
## select top
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']=[-np.log10(p) * 1 if act>1 else -np.log10(p) * -1 for act,p in zip(plotTab['act'],plotTab['padj'])]
## order by activity
plotTab=plotTab.sort_values(by='act',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

# make plot ---------------------
h,w=3,6

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='act'),color='black', stat='identity', show_legend=True,alpha=0.7) +
    p9.scale_fill_gradient2(low="#2166AC",mid="white",high="#B2182B",midpoint=0) +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.geom_hline(yintercept=np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_classic() +
    p9.labs(
        title=f"Top {topn} {pk}",
        subtitle='contrast: differential IFNG response (+,-) across disease groups (HR cases,HR controls) (interaction)\ndashed lines: padj 0.05',
        y='-log10(padj) * direction',
        fill='activity'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        axis_line_y=p9.element_blank(), axis_ticks_major_y=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / f"{pk}_overview.png", width=w, height=h, limitsize=False)
p.save(out_plots / f"{pk}_overview.pdf", width=w, height=h, limitsize=False)
plt.close()

/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 3 in image.
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /Users/miguel/Documents/Saezlab/project/olabisi_repo/olabisi/output/DEA/plots/funcomics_ulm/progeny_overview.png
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 3 in image.
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /Users/miguel/Documents/Saezlab/project/olabisi_repo/olabisi/output/DEA/plots/funcomics_ulm/progeny_overview.pdf


In [19]:
# save table with data to recreate plot

plotTab.drop(columns=['comparison']).rename(columns={'sig':'neglog10_padj_x_direction'}).to_csv(out_tab_progeny,sep='\t')

## Regulon of top TFs

- Plot the log2FC of top genes in top active TFs

#### **IRF1**

In [20]:
# # top TFs
# mask=((df_enrich['padj']<0.05) & (df_enrich['gene_set']=='collectri'))
# select_geneset=list(df_enrich[mask]['source'])

# ## for each geneset
# geneset=select_geneset[0]

# ## select top n features
# topn=12
# ### sort ascending by weight * stat
# ### the sign must be consistent (weight * stat = positive)
# select_features=df_collectri[df_collectri['source']==geneset].copy()
# select_features=(
#     select_features.
#     merge(deg.assign(target=lambda col:col['genesymbol'])[['target','stat']] ,how='left',on='target').
#     assign(score=lambda col:col['stat']*col['weight']).
#     sort_values(by='score',ascending=False).
#     head(topn)
#     ['target'].tolist()
# )

# ## prepare plot tab
# ptab=(
#     df_log2fc[df_log2fc['genesymbol'].isin(select_features)].copy().
#     # melt(id_vars='individualID',var_name='genesymbol',value_name='log2FC').
#     merge(adata.obs[adata.obs['condition_group']=='IFNG_pos'].reset_index()[['individualID','disease_group']],on='individualID',how='left')
# )
# ### factors
# col_dict={'HR_case':'HR case, IFNG+ vs IFNG-', 'HR_control':'HR control, IFNG+ vs IFNG-'}
# ptab['grp']=ptab['disease_group'].map(col_dict)
# ptab['grp']=ptab['grp'].astype('category').cat.reorder_categories(['HR case, IFNG+ vs IFNG-', 'HR control, IFNG+ vs IFNG-'],ordered=True)
# ptab['genesymbol']=ptab['genesymbol'].astype('category').cat.reorder_categories(select_features,ordered=True)
# ptab=ptab.sort_values(by=['genesymbol','grp'])

# ## add stat and padj
# cols=['stat','padj']
# for col in cols:
#     col_dict={x:y for x,y in zip(deg['genesymbol'],deg[col])}
#     ptab[col]=ptab['genesymbol'].map(col_dict)
# ptab['padj']=['{:.2e}'.format(x) for x in ptab['padj']]
# ptab['p_title']=[f"{a}\nStat: {round(float(b),2)}\nPadj: {c}" for a,b,c in zip(ptab['genesymbol'],ptab['stat'],ptab['padj'])]
# ptab['p_title']=ptab['p_title'].astype('category').cat.reorder_categories(ptab['p_title'].unique(), ordered=True)

# ptab

In [21]:
# # as boxplot

# ## palette
# pal = [c for c in sns.color_palette('tab10',n_colors=2).as_hex()] # blue, orange
# pal = pal[::-1]
# pal = {cat:color for cat,color in zip(ptab['grp'].cat.categories,pal)}

# plots_per_col=6
# nrows=ceil(len(select_features)/plots_per_col)
# w=2.3*plots_per_col if plots_per_col>=4 else 6
# h=3*nrows

# contrast='grp'
# p=(
#     p9.ggplot(ptab, p9.aes(x=contrast, y='log2_fold_change', group=contrast)) +
#     p9.geom_boxplot(outlier_alpha=0,size=0.8,width=0.5,show_legend=False) +
#     p9.geom_jitter(p9.aes(fill=contrast), color='black', size=2, width=0.1,alpha=1.0,show_legend=True) + 
#     p9.scale_fill_manual(pal) +
#     p9.guides(fill=p9.guide_legend(override_aes={'size':6})) +
#     p9.theme_classic() +
#     p9.labs(title=f"Log2 fold change of top {topn} gene targets for collecTRI {geneset}",y='Log2FC',x='disease group') +
#     p9.theme(
#         plot_title=p9.element_text(hjust=0.5, size=14), 
#         strip_text=p9.element_text(size=12), 
#         axis_title=p9.element_text(size=14), 
#         axis_text_x=p9.element_blank(), 
#         legend_text=p9.element_text(size=12),
#         figure_size=(w, h),
#     ) + 
#     p9.facet_wrap('p_title', scales = 'free', ncol=plots_per_col) 
# )

# p.save(out_plots / f"collectri_topTargets_{geneset}.png", width=w, height=h, limitsize=False)
# p.save(out_plots / f"collectri_topTargets_{geneset}.png", width=w, height=h, limitsize=False)
# plt.close()

#### **IRF1,NFKB1**

In [22]:
# mapping between geneset and output name to save table to recreate figure
out_dict={'IRF1':out_tab_collectri_IRF1,'NFKB1':out_tab_collectri_NFKB1} 

In [23]:
# top TFs
mask=((df_enrich['padj']<0.05) & (df_enrich['gene_set']=='collectri'))
select_geneset=list(df_enrich[mask]['source'])

## for each geneset
for geneset in select_geneset: # TF / progeny pathway
    
    # prepare data for plot --------------------------------------------------
    ## select top n features
    topn=18
    ### sort ascending by weight * stat
    ### the sign must be consistent (weight * stat = positive)
    select_features=df_collectri[df_collectri['source']==geneset].copy()
    select_features=(
        select_features.
        merge(deg.assign(target=lambda col:col['genesymbol'])[['target','stat']] ,how='left',on='target').
        assign(score=lambda col:col['stat']*col['weight']).
        sort_values(by='score',ascending=False).
        head(topn)
        ['target'].tolist()
    )

    ## prepare plot tab
    ptab=(
        df_log2fc[df_log2fc['genesymbol'].isin(select_features)].copy().
        merge(adata.obs[adata.obs['condition_group']=='IFNG_pos'].reset_index()[['individualID','disease_group']],on='individualID',how='left')
    )
    ### factors
    col_dict={'HR_case':'HR case, IFNG+ vs IFNG-', 'HR_control':'HR control, IFNG+ vs IFNG-'}
    ptab['grp']=ptab['disease_group'].map(col_dict)
    ptab['grp']=ptab['grp'].astype('category').cat.reorder_categories(['HR case, IFNG+ vs IFNG-', 'HR control, IFNG+ vs IFNG-'],ordered=True)
    ptab['genesymbol']=ptab['genesymbol'].astype('category').cat.reorder_categories(select_features,ordered=True)
    ptab=ptab.sort_values(by=['genesymbol','grp'])

    ## add stat and padj
    cols=['stat','padj']
    for col in cols:
        col_dict={x:y for x,y in zip(deg['genesymbol'],deg[col])}
        ptab[col]=ptab['genesymbol'].map(col_dict)
    ptab['padj']=['{:.2e}'.format(x) for x in ptab['padj']]
    ptab['p_title']=[f"{a}\nStat: {round(float(b),2)}\nPadj: {c}" for a,b,c in zip(ptab['genesymbol'],ptab['stat'],ptab['padj'])]
    ptab['p_title']=ptab['p_title'].astype('category').cat.reorder_categories(ptab['p_title'].unique(), ordered=True)

    ## save table with data to recreate plot
    ptab.to_csv(out_dict[geneset],sep='\t')


    # make plot -----------------------------------------------------------------
    # (boxplot)

    ## palette
    pal = [c for c in sns.color_palette('tab10',n_colors=2).as_hex()] # blue, orange
    pal = pal[::-1]
    pal = {cat:color for cat,color in zip(ptab['grp'].cat.categories,pal)}

    plots_per_col=6
    nrows=ceil(len(select_features)/plots_per_col)
    w=2.3*plots_per_col if plots_per_col>=4 else 6
    h=3*nrows

    contrast='grp'
    p=(
        p9.ggplot(ptab, p9.aes(x=contrast, y='log2_fold_change', group=contrast)) +
        p9.geom_boxplot(outlier_alpha=0,size=0.8,width=0.5,show_legend=False) +
        p9.geom_jitter(p9.aes(fill=contrast), color='black', size=2, width=0.1,alpha=1.0,show_legend=True) + 
        p9.scale_fill_manual(pal) +
        p9.guides(fill=p9.guide_legend(override_aes={'size':6})) +
        p9.theme_classic() +
        p9.labs(
            title=f"Log2 fold change of top {topn} gene targets for CollecTRI {geneset}",
            subtitle='Order: descending (collectri_weight * DEA_stat)\nOnly genes with consistent mode of regulation are included (e.g., positive collectri_weight * DEA_stat)',
            y='Log2FC',x='disease group'
        ) +
        p9.theme(
            plot_title=p9.element_text(hjust=0.5, size=14), 
            strip_text=p9.element_text(size=12), 
            axis_title=p9.element_text(size=14), 
            axis_text_x=p9.element_blank(), 
            legend_text=p9.element_text(size=12),
            figure_size=(w, h),
        ) + 
        p9.facet_wrap('p_title', scales = 'free', ncol=plots_per_col) 
    )
    p.save(out_plots / f"collectri_topTargets_{geneset}.png", width=w, height=h, limitsize=False)
    p.save(out_plots / f"collectri_topTargets_{geneset}.pdf", width=w, height=h, limitsize=False)
    plt.close()

/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 13.799999999999999 x 9 in image.
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /Users/miguel/Documents/Saezlab/project/olabisi_repo/olabisi/output/DEA/plots/funcomics_ulm/collectri_topTargets_IRF1.png
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 13.799999999999999 x 9 in image.
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /Users/miguel/Documents/Saezlab/project/olabisi_repo/olabisi/output/DEA/plots/funcomics_ulm/collectri_topTargets_IRF1.pdf
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 13.799999999999999 x 9 in image.
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10

## Top pathways

#### **Progeny**

In [24]:
# mapping between geneset and output name to save table to recreate figure
out_dict={
    'Hypoxia':out_tab_progeny_hypoxia,
    'NFkB':out_tab_progeny_NFKB,
    'TNFa':out_tab_progeny_TNFa,
    'JAK-STAT':out_tab_progeny_JAKSTAT
} 

In [25]:
# top TFs
mask=((df_enrich['padj']<0.05) & (df_enrich['gene_set']=='progeny'))
select_geneset=list(df_enrich[mask]['source'])

## for each geneset
for geneset in select_geneset: # TF / progeny pathway
    
    # prepare data for plot --------------------------------------------------
    ## select top n features
    topn=18
    ### sort ascending by weight * stat
    ### the sign must be consistent (weight * stat = positive)
    select_features=df_progeny[df_progeny['source']==geneset].copy()
    select_features=(
        select_features.
        merge(deg.assign(target=lambda col:col['genesymbol'])[['target','stat']] ,how='left',on='target').
        assign(score=lambda col:col['stat']*col['weight']).
        sort_values(by='score',ascending=False).
        head(topn)
        ['target'].tolist()
    )

    ## prepare plot tab
    ptab=(
        df_log2fc[df_log2fc['genesymbol'].isin(select_features)].copy().
        merge(adata.obs[adata.obs['condition_group']=='IFNG_pos'].reset_index()[['individualID','disease_group']],on='individualID',how='left')
    )
    ### factors
    col_dict={'HR_case':'HR case, IFNG+ vs IFNG-', 'HR_control':'HR control, IFNG+ vs IFNG-'}
    ptab['grp']=ptab['disease_group'].map(col_dict)
    ptab['grp']=ptab['grp'].astype('category').cat.reorder_categories(['HR case, IFNG+ vs IFNG-', 'HR control, IFNG+ vs IFNG-'],ordered=True)
    ptab['genesymbol']=ptab['genesymbol'].astype('category').cat.reorder_categories(select_features,ordered=True)
    ptab=ptab.sort_values(by=['genesymbol','grp'])

    ## add stat and padj
    cols=['stat','padj']
    for col in cols:
        col_dict={x:y for x,y in zip(deg['genesymbol'],deg[col])}
        ptab[col]=ptab['genesymbol'].map(col_dict)
    ptab['padj']=['{:.2e}'.format(x) for x in ptab['padj']]
    ptab['p_title']=[f"{a}\nStat: {round(float(b),2)}\nPadj: {c}" for a,b,c in zip(ptab['genesymbol'],ptab['stat'],ptab['padj'])]
    ptab['p_title']=ptab['p_title'].astype('category').cat.reorder_categories(ptab['p_title'].unique(), ordered=True)

    ## save table with data to recreate plot
    if geneset in out_dict.keys():
        ptab.to_csv(out_dict[geneset],sep='\t')

    # make plot -----------------------------------------------------------------
    # (boxplot)

    ## palette
    pal = [c for c in sns.color_palette('tab10',n_colors=2).as_hex()] # blue, orange
    pal = pal[::-1]
    pal = {cat:color for cat,color in zip(ptab['grp'].cat.categories,pal)}

    plots_per_col=6
    nrows=ceil(len(select_features)/plots_per_col)
    w=2.3*plots_per_col if plots_per_col>=4 else 6
    h=3*nrows

    contrast='grp'
    p=(
        p9.ggplot(ptab, p9.aes(x=contrast, y='log2_fold_change', group=contrast)) +
        p9.geom_boxplot(outlier_alpha=0,size=0.8,width=0.5,show_legend=False) +
        p9.geom_jitter(p9.aes(fill=contrast), color='black', size=2, width=0.1,alpha=1.0,show_legend=True) + 
        p9.scale_fill_manual(pal) +
        p9.guides(fill=p9.guide_legend(override_aes={'size':6})) +
        p9.theme_classic() +
        p9.labs(title=f"Log2 fold change of top {topn} gene targets for Progeny {geneset}",y='Log2FC',x='disease group') +
        p9.theme(
            plot_title=p9.element_text(hjust=0.5, size=14), 
            strip_text=p9.element_text(size=12), 
            axis_title=p9.element_text(size=14), 
            axis_text_x=p9.element_blank(), 
            legend_text=p9.element_text(size=12),
            figure_size=(w, h),
        ) + 
        p9.facet_wrap('p_title', scales = 'free', ncol=plots_per_col) 
    )
    p.save(out_plots / f"progeny_topTargets_{geneset}.png", width=w, height=h, limitsize=False)
    p.save(out_plots / f"progeny_topTargets_{geneset}.pdf", width=w, height=h, limitsize=False)
    plt.close()

/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 13.799999999999999 x 9 in image.
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /Users/miguel/Documents/Saezlab/project/olabisi_repo/olabisi/output/DEA/plots/funcomics_ulm/progeny_topTargets_Hypoxia.png
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 13.799999999999999 x 9 in image.
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /Users/miguel/Documents/Saezlab/project/olabisi_repo/olabisi/output/DEA/plots/funcomics_ulm/progeny_topTargets_Hypoxia.pdf
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 13.799999999999999 x 9 in image.
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.

## Selected Genes

In [26]:
## prepare data for plot

select_features=[
    "APOL1", "APOL6", "IRF1", "STAT1", "STAT2", "STAT3",
    "IRF2", "IRF8", "IRF9", "CCL2", "CXCL9", "CXCL10",
    "CXCL11", "ICAM1", "IDO1", "GBP1", "PSMB9", "TMEM140", "CTSS",
]

## prepare plot tab
ptab=(
    df_log2fc[df_log2fc['genesymbol'].isin(select_features)].copy().
    merge(adata.obs[adata.obs['condition_group']=='IFNG_pos'].reset_index()[['individualID','disease_group']],on='individualID',how='left')
)
### factors
col_dict={'HR_case':'HR case, IFNG+ vs IFNG-', 'HR_control':'HR control, IFNG+ vs IFNG-'}
ptab['grp']=ptab['disease_group'].map(col_dict)
ptab['grp']=ptab['grp'].astype('category').cat.reorder_categories(['HR case, IFNG+ vs IFNG-', 'HR control, IFNG+ vs IFNG-'],ordered=True)
ptab['genesymbol']=ptab['genesymbol'].astype('category').cat.reorder_categories(select_features,ordered=True)
ptab=ptab.sort_values(by=['genesymbol','grp'])

## add stat and padj
cols=['stat','padj']
for col in cols:
    col_dict={x:y for x,y in zip(deg['genesymbol'],deg[col])}
    ptab[col]=ptab['genesymbol'].map(col_dict)
ptab['padj']=['{:.2e}'.format(x) for x in ptab['padj']]
ptab['p_title']=[f"{a}\nStat: {round(float(b),2)}\nPadj: {c}" for a,b,c in zip(ptab['genesymbol'],ptab['stat'],ptab['padj'])]
ptab['p_title']=ptab['p_title'].astype('category').cat.reorder_categories(ptab['p_title'].unique(), ordered=True)

## save table with data to recreate plot
ptab.to_csv(out_tab_customGenes,sep='\t')

ptab.head()

,individualID,genesymbol,IFNG_neg,IFNG_pos,fold_change,log2_fold_change,disease_group,grp,stat,padj,p_title
0,individual_01,APOL1,14.334884,1782.217875,124.327326,6.958000,HR_case,"HR case, IFNG+ vs IFNG-",1.063783,9.89e-01,APOL1\nStat: 1.06\nPadj: 9.89e-01
19,individual_02,APOL1,5.123411,226.366974,44.182857,5.465415,HR_case,"HR case, IFNG+ vs IFNG-",1.063783,9.89e-01,APOL1\nStat: 1.06\nPadj: 9.89e-01
38,individual_03,APOL1,18.323577,1529.368002,83.464485,6.383091,HR_case,"HR case, IFNG+ vs IFNG-",1.063783,9.89e-01,APOL1\nStat: 1.06\nPadj: 9.89e-01
57,individual_04,APOL1,4.891112,290.383022,59.369519,5.891651,HR_case,"HR case, IFNG+ vs IFNG-",1.063783,9.89e-01,APOL1\nStat: 1.06\nPadj: 9.89e-01
76,individual_05,APOL1,40.231772,3520.192049,87.497811,6.451175,HR_case,"HR case, IFNG+ vs IFNG-",1.063783,9.89e-01,APOL1\nStat: 1.06\nPadj: 9.89e-01


In [27]:
# make plot -----------------------------------------------------------------
# (boxplot)

## palette
pal = [c for c in sns.color_palette('tab10',n_colors=2).as_hex()] # blue, orange
pal = pal[::-1]
pal = {cat:color for cat,color in zip(ptab['grp'].cat.categories,pal)}

plots_per_col=6
nrows=ceil(len(select_features)/plots_per_col)
w=2.3*plots_per_col if plots_per_col>=4 else 6
h=3*nrows

contrast='grp'
p=(
    p9.ggplot(ptab, p9.aes(x=contrast, y='log2_fold_change', group=contrast)) +
    p9.geom_boxplot(outlier_alpha=0,size=0.8,width=0.5,show_legend=False) +
    p9.geom_jitter(p9.aes(fill=contrast), color='black', size=2, width=0.1,alpha=1.0,show_legend=True) + 
    p9.scale_fill_manual(pal) +
    p9.guides(fill=p9.guide_legend(override_aes={'size':6})) +
    p9.theme_classic() +
    p9.labs(title=f"Log2 fold change of selected genes",y='Log2FC',x='disease group') +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=14), 
        strip_text=p9.element_text(size=12), 
        axis_title=p9.element_text(size=14), 
        axis_text_x=p9.element_blank(), 
        legend_text=p9.element_text(size=12),
        figure_size=(w, h),
    ) + 
    p9.facet_wrap('p_title', scales = 'free', ncol=plots_per_col) 
)
p.save(out_plots / "customGenes.png", width=w, height=h, limitsize=False)
p.save(out_plots / "customGenes.pdf", width=w, height=h, limitsize=False)
plt.close()

/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 13.799999999999999 x 12 in image.
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /Users/miguel/Documents/Saezlab/project/olabisi_repo/olabisi/output/DEA/plots/funcomics_ulm/customGenes.png
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 13.799999999999999 x 12 in image.
/Users/miguel/miniforge3/envs/squidpy_v09/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /Users/miguel/Documents/Saezlab/project/olabisi_repo/olabisi/output/DEA/plots/funcomics_ulm/customGenes.pdf
